# MOSS-Image Colab Studio

**開發中（規格 v1.3.0）**：文生圖與排版分頁可用；其他分頁依 Stage 3～8 陸續開放。未通過 GPU 驗收的功能會在介面標示，不會假裝可用。

使用前提：付費 Colab、維持正的 compute unit 餘額、執行階段選 **L4 GPU** 與高 RAM。工具不會替你購買資源或掛載 Google Drive。模型約 20 GiB，存在 Colab 本地；VM 回收後本地檔案與作品會消失，請自行下載作品或專案。

連線：`share=False`，ComfyUI 只綁 loopback；對外透過 Cloudflare Quick Tunnel，並以本次產生的一次性密碼登入。

**使用限制**：不得以真人照片製作換臉、私密或冒用他人身分的內容（Colab 對所有執行環境禁止 deepfakes）。

In [ ]:
# @title 1. 取得程式（預設 main；可填固定 commit）
SOURCE_REVISION = "main" # @param {type:"string"}
import os, sys, subprocess, pathlib, json
SOURCE = pathlib.Path('/content/MOSS-Image-Colab')
if not (SOURCE / '.git').exists():
    subprocess.run(['git','clone','https://github.com/Openclaw-Metis/MOSS-Image-Colab.git',str(SOURCE)],check=True)
if subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=SOURCE,text=True).strip():
    raise RuntimeError('原始碼有本地修改，保留現況；請先另存後再重新啟動。')
subprocess.run(['git','fetch','origin',SOURCE_REVISION],cwd=SOURCE,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=SOURCE,check=True)
SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=SOURCE,text=True).strip()
sys.path.insert(0,str(SOURCE/'src'))
os.environ['PYTHONPATH']=str(SOURCE/'src')
print('Resolved source commit:',SOURCE_COMMIT)

In [ ]:
# @title 2. 檢查 L4、安裝固定環境、下載並校驗模型、啟動引擎
from moss_studio.services.bootstrap import ColabBootstrap
from moss_studio.services.resources import inspect_runtime
bootstrap = ColabBootstrap(SOURCE)
print(json.dumps(inspect_runtime(bootstrap.root),indent=2))
bootstrap.prepare(allow_candidate=True)
bootstrap.download(['sft_bf16','qwen_int8','vae_small'])
engine_state=bootstrap.start_engine()
print('ComfyUI:',engine_state)

In [ ]:
# @title 3. 啟動創作工作台（loopback，需密碼登入）
import secrets, urllib.request
from moss_studio.services.processes import ManagedProcess
DATA_ROOT = pathlib.Path('/content/moss-studio')
if 'studio_password' not in globals():
    studio_password=secrets.token_urlsafe(24)
os.environ.update(MOSS_STUDIO_PASSWORD=studio_password, GRADIO_ANALYTICS_ENABLED='False',
                  GRADIO_TEMP_DIR=str(DATA_ROOT/'gradio_cache'))
studio_app=ManagedProcess(bootstrap.root/'.runtime','studio_app')
def app_health():
    with urllib.request.urlopen('http://127.0.0.1:7860/login',timeout=3) as response:
        if response.status != 200: raise RuntimeError('App not ready')
try:
    app_state=studio_app.start([str(bootstrap.python),'-m','moss_studio.ui.app','--repository',str(SOURCE),
                                '--data-root',str(DATA_ROOT)],SOURCE,7860,app_health,timeout=120)
finally:
    os.environ.pop('MOSS_STUDIO_PASSWORD',None)
print('App PID:',app_state['pid'],'reused:',app_state['reused'])
print('本次入口密碼：',studio_password)
print('這是暫時密碼；不要儲存或分享含密碼的 Notebook 輸出。')

In [ ]:
# @title 4. 啟動固定版本 Quick Tunnel（不使用 Gradio share）
import hashlib, re
runtime=json.loads((SOURCE/'configs/runtime.lock.json').read_text())
binary=bootstrap.root/'cloudflared'; pinned=runtime['cloudflared']
if not binary.exists() or hashlib.sha256(binary.read_bytes()).hexdigest()!=pinned['sha256']:
    with urllib.request.urlopen(pinned['url'],timeout=60) as response: data=response.read(100*1024**2+1)
    if len(data)>100*1024**2 or hashlib.sha256(data).hexdigest()!=pinned['sha256']:
        raise RuntimeError('cloudflared 校驗失敗，不啟動')
    partial=binary.with_suffix('.part'); partial.write_bytes(data); partial.chmod(0o700); partial.replace(binary)
tunnel=ManagedProcess(bootstrap.root/'.runtime','tunnel')
def tunnel_health():
    with urllib.request.urlopen('http://127.0.0.1:20242/ready',timeout=3) as response:
        if response.status!=200: raise RuntimeError('Tunnel not ready')
state=tunnel.start([str(binary),'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate','--protocol','http2',
                    '--metrics','127.0.0.1:20242'],SOURCE,20242,tunnel_health)
urls=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',tunnel.log.read_text())
if not urls: raise RuntimeError('通道未回報網址；不退回 share=True')
print('創作工作台：',urls[-1]+'/login')
print('用上一格產生的密碼登入。')

## 操作說明

登入後在「文生圖與排版」輸入描述，按「產生圖片」。勾選「啟用畫框排版」可拖曳文字、物件與人物框。進度顯示目前階段與步數；完成後可下載圖片或打包作品 ZIP。重新整理頁面會找回原工作，不會自動重送。

需要結束時：等工作完成或取消，下載要保留的作品，再執行最後一格停止工具。

In [ ]:
# @title 5.（驗收用，預設關閉）GPU 批次 A：AC-04 基準與 AC-19 繁中排版
# 執行前請讓工作台閒置。腳本每完成一張就寫入檢查點；中斷後重跑此格會接續，不重送已完成的工作。
RUN_BATCH_A = False # @param {type:"boolean"}
EVIDENCE = pathlib.Path('/content/moss-evidence')
if RUN_BATCH_A:
    env={**os.environ,'PYTHONPATH':str(SOURCE/'src')}
    for script,target in [('benchmark_http.py','stage2'),('zh_tw_layout_eval.py','zh_tw')]:
        env['MOSS_STUDIO_PASSWORD']=studio_password
        subprocess.run([str(bootstrap.python),str(SOURCE/'scripts'/script),str(EVIDENCE/target)],
                       cwd=SOURCE,env=env,check=True)
    import shutil
    archive=shutil.make_archive('/content/moss-evidence-batch-a','zip',EVIDENCE)
    print('證據已打包：',archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as error:
        print('請從左側檔案面板手動下載：',error)

In [ ]:
# @title 6.（驗收用，預設關閉）GPU 批次 B：VRAM 模式對照與 AC-06 加速 8 步
# 會依序以 auto 與 lowvram 重啟引擎各跑 1 筆 quality50，再以較快模式跑 fast8 暖機＋5 筆。
# 執行前請讓工作台閒置；結果寫入 /content/moss-evidence 並打包下載。修圖 AC-08 需在介面以固定素材操作。
RUN_BATCH_B = False # @param {type:"boolean"}
if RUN_BATCH_B:
    import shutil, time
    bootstrap.download(['dmd8_lora','edit_sft'])
    app_command=[str(bootstrap.python),'-m','moss_studio.ui.app','--repository',str(SOURCE),'--data-root',str(DATA_ROOT)]
    def restart(mode):
        studio_app.stop(); bootstrap.engine.stop()
        bootstrap.start_engine(vram_mode=mode)
        os.environ['MOSS_STUDIO_PASSWORD']=studio_password
        try: studio_app.start(app_command,SOURCE,7860,app_health,timeout=120)
        finally: os.environ.pop('MOSS_STUDIO_PASSWORD',None)
    def bench(target,*extra):
        env={**os.environ,'PYTHONPATH':str(SOURCE/'src'),'MOSS_STUDIO_PASSWORD':studio_password}
        subprocess.run([str(bootstrap.python),str(SOURCE/'scripts/benchmark_http.py'),str(EVIDENCE/target),*extra],
                       cwd=SOURCE,env=env,check=True)
    timings={}
    for mode in ['auto','lowvram']:
        restart(mode); bench(f'vram_{mode}','--count','1')
        report=json.loads((EVIDENCE/f'vram_{mode}'/'benchmark.json').read_text())
        timings[mode]=[s['record']['seconds'] for s in report['samples'] if s['status']=='succeeded']
    print('VRAM 模式耗時（暖機, 正式）：',timings)
    best=min(timings,key=lambda m:timings[m][-1]) if all(timings.values()) else 'lowvram'
    restart(best); bench('fast8','--profile','fast8_v1','--count','5')
    (EVIDENCE/'vram_choice.json').write_text(json.dumps({'timings':timings,'chosen':best},indent=2))
    archive=shutil.make_archive('/content/moss-evidence-batch-b','zip',EVIDENCE)
    print('證據已打包：',archive,'；請把 runtime.lock.json 的 engine_vram_mode 改為',best,'並回報結果')
    try:
        from google.colab import files
        files.download(archive)
    except Exception as error:
        print('請從左側檔案面板手動下載：',error)

In [ ]:
# @title 7. 停止工具（不刪除作品與模型；預設不執行）
STOP_STUDIO = False # @param {type:"boolean"}
if STOP_STUDIO:
    for name in ['tunnel','studio_app']:
        if name in globals(): globals()[name].stop()
    bootstrap.engine.stop()
    print('已停止本工具程序；作品與模型保留在本地。')